In [0]:
# =============================================================================
# Notebook  : 00_check_new_files
# Location  : /HGI-Lakehouse-Pipeline/00_Orchestration/00_check_new_files
# Purpose   : GATE TASK — checks whether new files arrived in the incremental
#             landing zone for BOTH Salesforce and BigQuery.
#
#             Sets a Databricks task value:
#               dbutils.jobs.taskValues.set("new_files_found", "true")
#               or
#               dbutils.jobs.taskValues.set("new_files_found", "false")
#
#             Downstream tasks in master_jobs.yml use run_if:
#               tasks['check_new_files'].values['new_files_found'] == 'true'
#
#             This means if NO new files arrived:
#               - Bronze tasks are SKIPPED automatically
#               - Silver tasks are SKIPPED automatically
#               - The job finishes immediately with no wasted compute
#
# HOW IT WORKS:
#   1. Scans S3 landing-zone/salesforce/{customer_id}/*/incremental/
#   2. Scans S3 landing-zone/bigquery/{customer_id}/events/incremental/
#   3. Uses Auto Loader's checkpoint metadata to find UNPROCESSED files only
#   4. If any unprocessed files exist → new_files_found = "true"
#   5. If no unprocessed files → new_files_found = "false"
#
# Serverless: works on Serverless (uses dbutils.fs.ls, no spark.conf issues)
# Job Compute: same code, same result
# =============================================================================


In [0]:
# CELL 1
%run ../config/pipeline_config

In [0]:
# CELL 2 ── Widgets
dbutils.widgets.text("customer_id", "cust_002")
customer_id = dbutils.widgets.get("customer_id").strip().lower()

from datetime import datetime

print(f"=== Gate: Check New Files ===")
print(f"  customer_id : {customer_id}")
print(f"  Checked at  : {datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S UTC')}")
print(f"  S3 root     : {landing_root}")

In [0]:
# CELL 3 ── Helper functions

def count_all_files_in_tree(path: str, depth: int = 0, max_depth: int = 5) -> int:
    """
    Recursively count ALL parquet files under a path (up to max_depth).
    Used for reporting total files in landing zone.
    """
    if depth > max_depth:
        return 0
    total = 0
    try:
        entries = dbutils.fs.ls(path)
        for entry in entries:
            if entry.name.endswith(".parquet") or entry.name.endswith(".snappy.parquet"):
                total += 1
            elif entry.isDir() and not entry.name.startswith("_"):
                total += count_all_files_in_tree(entry.path, depth + 1, max_depth)
    except Exception:
        pass
    return total


def get_latest_checkpoint_commit_ms(layer: str, source_sys: str, obj_name: str) -> float:
    """
    Returns the modification time (epoch ms) of the latest Auto Loader
    checkpoint commit file.  Returns 0.0 if no checkpoint exists (first run).
    """
    chk = checkpoint_path(layer, source_sys, customer_id, obj_name)
    try:
        commits_path = chk.rstrip("/") + "/commits/"
        commits = dbutils.fs.ls(commits_path)
        if not commits:
            return 0.0
        return max(c.modificationTime for c in commits)
    except Exception:
        return 0.0


def count_new_files_in_tree(path: str, since_ms: float = 0.0,
                            depth: int = 0, max_depth: int = 5) -> int:
    """
    Recursively count parquet files under a path that are NEWER than since_ms.
    If since_ms is 0.0 (no checkpoint / first run), counts ALL files.
    """
    if depth > max_depth:
        return 0
    total = 0
    try:
        entries = dbutils.fs.ls(path)
        for entry in entries:
            if entry.name.endswith(".parquet") or entry.name.endswith(".snappy.parquet"):
                if entry.modificationTime > since_ms:
                    total += 1
            elif entry.isDir() and not entry.name.startswith("_"):
                total += count_new_files_in_tree(entry.path, since_ms, depth + 1, max_depth)
    except Exception:
        pass
    return total

In [0]:
# CELL 4 ── Check Salesforce incremental landing zone
SF_OBJECTS = ["account", "contact", "opportunity", "task",
              "campaign", "campaignmember", "user"]

print(f"\n── Salesforce incremental files ──")
sf_new_total = 0
sf_all_total = 0
sf_details   = {}

for obj in SF_OBJECTS:
    path      = landing_path("salesforce", customer_id, obj, "incremental")
    all_count = count_all_files_in_tree(path)
    chkpt_ms  = get_latest_checkpoint_commit_ms("bronze", "salesforce", obj)
    new_count = count_new_files_in_tree(path, since_ms=chkpt_ms) if all_count > 0 else 0

    sf_details[obj] = {"all": all_count, "new": new_count, "chkpt_ms": chkpt_ms}
    sf_all_total += all_count
    sf_new_total += new_count

    chkpt_info = ("no checkpoint (first run)" if chkpt_ms == 0.0
                  else f"checkpoint @ {datetime.utcfromtimestamp(chkpt_ms / 1000).strftime('%Y-%m-%d %H:%M')}")
    icon = "?❗" if new_count > 0 else "─"
    print(f"  {icon}  {obj:<20} {new_count:>5} new / {all_count:>5} total  |  {chkpt_info}")

In [0]:
# CELL 5 ── Check BigQuery incremental landing zone
print(f"\n── BigQuery incremental files ──")
bq_path      = landing_path("bigquery", customer_id, "events", "incremental")
bq_all_total = count_all_files_in_tree(bq_path)
bq_chkpt_ms  = get_latest_checkpoint_commit_ms("bronze", "bigquery", "events")
bq_new_total = count_new_files_in_tree(bq_path, since_ms=bq_chkpt_ms) if bq_all_total > 0 else 0

chkpt_info = ("no checkpoint (first run)" if bq_chkpt_ms == 0.0
              else f"checkpoint @ {datetime.utcfromtimestamp(bq_chkpt_ms / 1000).strftime('%Y-%m-%d %H:%M')}")
icon = "?❗" if bq_new_total > 0 else "─"
print(f"  {icon}  {'events':<20} {bq_new_total:>5} new / {bq_all_total:>5} total  |  {chkpt_info}")

In [0]:
# CELL 6 ── Summary: which objects have truly NEW unprocessed files
print(f"\n── New-file summary (checkpoint-aware) ──")

for obj in SF_OBJECTS:
    d    = sf_details[obj]
    icon = "✅ NEW" if d["new"] > 0 else "⏸  none"
    print(f"  {icon}  salesforce/{obj:<20}  {d['new']:>5} new files")

bq_icon = "✅ NEW" if bq_new_total > 0 else "⏸  none"
print(f"  {bq_icon}  bigquery/events            {bq_new_total:>5} new files")

In [0]:
# CELL 7 ── (reserved — logic moved to checkpoint-aware cells above)

In [0]:
# CELL 8 ── Final gate decision  (based on NEW files only, not all files)
total_new_files = sf_new_total + bq_new_total
new_files_found = total_new_files > 0

# Keep sf_total / bq_total aliases for Cell 9 task-value compatibility
sf_total = sf_new_total
bq_total = bq_new_total

print(f"\n── Gate Decision ──")
print(f"  SF files in landing zone  : {sf_all_total:,} total, {sf_new_total:,} NEW")
print(f"  BQ files in landing zone  : {bq_all_total:,} total, {bq_new_total:,} NEW")
print(f"  Files to process          : {total_new_files:,}")
print()
if new_files_found:
    print(f"  ✅ NEW FILES FOUND → Setting new_files_found = 'true'")
    print(f"     All downstream bronze + silver tasks will RUN")
else:
    print(f"  ⏸  NO NEW FILES → Setting new_files_found = 'false'")
    print(f"     All downstream tasks will be SKIPPED (saves cost)")

In [0]:
# CELL 9 ── Set task value — this is what downstream run_if conditions read
# This is a Databricks-native feature: dbutils.jobs.taskValues
# Works in Jobs only (raises error in interactive mode — handled below)
try:
    dbutils.jobs.taskValues.set(key="new_files_found", value=str(new_files_found).lower())
    dbutils.jobs.taskValues.set(key="sf_files_count",  value=str(sf_total))
    dbutils.jobs.taskValues.set(key="bq_files_count",  value=str(bq_total))
    print(f"\n  Task values set:")
    print(f"    new_files_found = '{str(new_files_found).lower()}'")
    print(f"    sf_files_count  = '{sf_total}'")
    print(f"    bq_files_count  = '{bq_total}'")
except Exception as e:
    # Running interactively (not inside a job) — task values not available
    print(f"\n  NOTE: Running interactively — task values not set (only works inside a Job)")
    print(f"  new_files_found would be: {str(new_files_found).lower()}")

In [0]:
# CELL 10 ── Exit with the result (used by the calling job for run_if)
dbutils.notebook.exit(str(new_files_found).lower())   # "true" or "false"